# Getting Started with Coordinatus

This notebook introduces the core concepts of **Coordinatus**:
- What a `Space` is and how spaces relate to each other
- `Point` vs `Vector`: why the distinction matters
- Converting coordinates between spaces using `.relative_to()` and `.to_absolute()`
- A practical example: temperature unit conversion


## Installation

If you haven't installed Coordinatus yet:
```bash
uv add coordinatus
```

## 1. Points and Vectors

A `Point` represents a position — it is affected by translation, rotation, and scale.

A `Vector` represents a direction — it is **not** affected by translation.

In [1]:
import numpy as np
from coordinatus import Space2D, Space, Point, Vector, create_space

# Create a child space offset by (10, 5) relative to the world root
world = Space2D()
frame = create_space(parent=world, tx=10, ty=5, angle_rad=np.pi / 4)

# A point at the origin of `frame`
point = Point([0, 0], space=frame)
print("Point in frame:    ", point.coords)
print("Point in world:    ", point.to_absolute().coords.round(6))

# A unit vector along the local X axis
vector = Vector([1, 0], space=frame)
print("\nVector in frame:   ", vector.coords)
print("Vector in world:   ", vector.to_absolute().coords.round(6))
print("(translation has no effect on the vector — only rotation does)")


Point in frame:     [0 0]
Point in world:     [10.  5.]

Vector in frame:    [1 0]
Vector in world:    [0.707107 0.707107]
(translation has no effect on the vector — only rotation does)


## 2. Hierarchical Spaces

Spaces can be nested. Converting from a deeply nested space to the world requires walking up the entire chain.

In [2]:
# car → wheel hierarchy
car   = create_space(parent=world, tx=100, ty=50, angle_rad=np.pi / 4)
wheel = create_space(parent=car,   tx=10,  ty=0)

# A point at the wheel's local origin
p = Point([0, 0], space=wheel)
print("Wheel origin in world space:", p.to_absolute().coords.round(4))

# Express the same point in car space (parent)
p_in_car = p.relative_to(car)
print("Wheel origin in car space:  ", p_in_car.coords.round(4))


Wheel origin in world space: [107.0711  57.0711]
Wheel origin in car space:   [10.  0.]


## 3. Temperature Scales — A 1D Example

Temperature unit systems are just 1D spaces linked by translation and scaling. No physics formula needed — define the spaces, and Coordinatus handles the rest.

In [3]:
from coordinatus import Space1D, Point
from coordinatus.transforms import translate1D, scale1D

# Kelvin is the root (absolute) 1D space
kelvin     = Space1D()
celsius    = Space(transform=translate1D(273.15),                parent=kelvin)
fahrenheit = Space(transform=scale1D(5/9) @ translate1D(-32),   parent=celsius)

temperatures = {
    "Absolute zero":          Point([0.0],   space=kelvin),
    "Freezing point":         Point([0.0],   space=celsius),
    "Human body (~100 °F)":   Point([100.0], space=fahrenheit),
    "Boiling point":          Point([100.0], space=celsius),
}

header = f"{'Temperature':<26} {'Kelvin':>10} {'Celsius':>10} {'Fahrenheit':>12}"
print(header)
print("-" * len(header))
for name, pt in temperatures.items():
    k = pt.relative_to(kelvin).coords[0]
    c = pt.relative_to(celsius).coords[0]
    f = pt.relative_to(fahrenheit).coords[0]
    print(f"{name:<26} {k:>9.2f}K {c:>9.2f}°C {f:>10.2f}°F")


Temperature                    Kelvin    Celsius   Fahrenheit
-------------------------------------------------------------
Absolute zero                   0.00K   -273.15°C    -459.67°F
Freezing point                273.15K      0.00°C      32.00°F
Human body (~100 °F)          310.93K     37.78°C     100.00°F
Boiling point                 373.15K    100.00°C     212.00°F


## 4. Arithmetic on Coordinates

Coordinates in the **same space** support element-wise arithmetic.

In [4]:
a = Point([3, 4], space=world)
b = Point([1, 2], space=world)

print("a =", a.coords)
print("b =", b.coords)
print("a + b =", (a + b).coords)
print("a - b =", (a - b).coords)
print("a * 2 =", (a * 2).coords)

# Vectors can be added to points
v = Vector([0, 1], space=world)
print("a + v =", (a + v).coords)


a = [3 4]
b = [1 2]
a + b = [4 6]
a - b = [2 2]
a * 2 = [6 8]
a + v = [3 5]
